In [2]:
import sys
import os
from pathlib import Path

ROOT_DIR = Path("..").resolve()
os.chdir(ROOT_DIR)

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data_loader import load_data
from src.cleaning import DataCleaner

# Load and clean dataset
df_raw = load_data()
df = (
    DataCleaner(df_raw)
    .remove_duplicates()
    .clean_gender()
    .clean_city()
    .clean_monthly_charges()
    .clean_dates()
    .clean_negative_values()
    .fill_missing()
    .df
)

In [3]:
# Separate features and target variable
TARGET = "churn"
X = df.drop(columns=[TARGET, "customer_id", "full_name", "email", "phone", 
                     "date_of_birth", "customer_since", "last_login_date", "last_payment_date"], errors="ignore")
y = df[TARGET].map({"No": 0, "Yes": 1})

# Define feature categories
numeric_features = ["monthly_charges", "total_charges", "tenure_months", "support_tickets", "complaint_count", "avg_monthly_usage_gb"]
categorical_features = ["gender", "city", "contract_type", "subscription_type", "internet_service"]

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

Feature matrix shape: (200000, 13)
Target distribution:
churn
0    0.649305
1    0.350695
Name: proportion, dtype: float64


In [4]:
# Perform 80/20 train-test split preventing data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")

Training set: (160000, 13)
Testing set:  (40000, 13)


In [5]:
# Build Scikit-Learn ColumnTransformer
numeric_pipeline = Pipeline([("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Fit preprocessor on training data only
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

# Save pipeline artifact
os.makedirs("models", exist_ok=True)
joblib.dump(preprocessor, "models/preprocessor.pkl")
print("Saved fitted ColumnTransformer to models/preprocessor.pkl successfully!")

Saved fitted ColumnTransformer to models/preprocessor.pkl successfully!
